# Name That LLM — PACT-18 提交 Notebook

**需要的输入（Add Input）**

1. `Competitions` → **Name That LLM**
2. `Datasets` → 本仓库上传后的数据集（需包含 `src/` 与 `configs/`）
3. `Models` → **Qwen3-8B-Base**（4-bit 推理；缺失时脚本会退回 4 成员配置并在日志中告警）

**运行设置**：Accelerator = `GPU T4 x2`，Internet = `Off`。
**产物**：`/kaggle/working/submission.parquet`（两列 `id, model`）。


In [ ]:
import json, sys, time, zipfile
from pathlib import Path

def _first(cands):
    for c in cands:
        p = Path(c)
        if p.exists():
            return p
    return None

COMP = _first(['/kaggle/input/name-that-llm', '/kaggle/input/competitions/name-that-llm'])
assert COMP is not None and (COMP / 'test.csv').exists(), 'Add Input -> Competition: Name That LLM'

def _find_src(root):
    for pkg in Path(root).rglob('ntl_pact18'):
        if pkg.is_dir() and (pkg / 'pipeline.py').exists():
            return pkg.parent
    return None

SRC = _find_src('/kaggle/input')
if SRC is None:
    zips = sorted(z for z in Path('/kaggle/input').rglob('*.zip') if 'ntl-pact18' in z.name)
    assert zips, 'Add Input -> Dataset: ntl-pact18 source (zip)'
    dest = Path('/kaggle/working/code')
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(dest)
    SRC = _find_src(dest)
assert SRC is not None, 'could not locate ntl_pact18 package'
CODE = SRC.parent

QWEN = None
for cfg in Path('/kaggle/input').rglob('config.json'):
    try:
        meta = json.loads(cfg.read_text())
    except Exception:
        continue
    if str(meta.get('model_type', '')).startswith('qwen3'):
        QWEN = cfg.parent
        break

print('competition :', COMP)
print('code        :', CODE)
print('qwen        :', QWEN)


In [ ]:
sys.path.insert(0, str(CODE / 'src'))

from ntl_pact18.config import load_config
from ntl_pact18.data import load_train
from ntl_pact18.pipeline import PACT18Pipeline
from ntl_pact18.runtime import build_profile, describe

profile = build_profile('kaggle')
print(describe(profile))
cfg = load_config(str(CODE / 'configs' / 'real.yaml'), profile='kaggle')

# preflight: 4-bit Qwen needs torch + transformers + bitsandbytes on a CUDA device
QWEN_OK = False
if profile.qwen_enabled:
    try:
        import torch, transformers, bitsandbytes
        QWEN_OK = torch.cuda.is_available()
        print('transformers', transformers.__version__, '| cuda', torch.cuda.is_available())
    except Exception as exc:
        print('[warn] qwen deps unavailable:', repr(exc))

if not QWEN_OK:
    print('[warn] Qwen3-8B not usable: falling back to the 4-member stack (lower score)')
    cfg.set('features.qwen.enabled', False)

train = load_train(COMP / 'train.csv')
print('train rows =', len(train), '| prompts =', train['prompt_id'].nunique(), '| labels =', train['model'].nunique())

t0 = time.time()
pipe = PACT18Pipeline(cfg).fit(train, verbose=True)
print('fit seconds =', round(time.time() - t0, 1))
pipe.save(Path('/kaggle/working/artifacts/model.joblib'))


In [ ]:
import pandas as pd
from ntl_pact18.data import load_test

test = load_test(COMP / 'test.csv')
print('test rows =', len(test))
pred = pipe.predict(test, verbose=True)

sub = pd.DataFrame({'id': test['id'].astype(str), 'model': pred.labels()})
sub.to_parquet('/kaggle/working/submission.parquet', index=False)
print(sub.head())


In [ ]:
from ntl_pact18.labels import LABELS

assert list(sub.columns) == ['id', 'model'], sub.columns
assert len(sub) == len(test)
assert sub['id'].is_unique and sub['id'].notna().all()
assert sub['model'].notna().all()
assert set(sub['id']) == set(test['id'].astype(str))
assert set(sub['model']).issubset(set(LABELS)), sorted(set(sub['model']) - set(LABELS))
print('rows =', len(sub), '| unique ids =', sub['id'].nunique(), '| labels used =', sub['model'].nunique())
print(sub['model'].value_counts().to_string())


提交前检查：`submission.parquet` 行数应等于 `test.csv` 行数，且 `model` 全部落在 18 个标签内。
Kaggle 会重置到隐藏测试集（约 4,000 行）重跑本 notebook，因此这里的分支只用于自检。
